# Computer Vision - Assignment 2
## Problem Statement 3: Advanced Object Tracking and Detection in Video Streams
**Course:** Computer Vision (S2-25_AIMLCZG525) &nbsp;&nbsp;|&nbsp;&nbsp; **Group:** 12

---

### Objective
Build an **advanced object tracking and detection system** that uses a **Faster R-CNN**
detector to identify and track multiple objects (**sports players**) across the frames of a
video stream and enhance robustness with two novel techniques:

1. **Temporal consistency checks** - a track is only confirmed after it is detected over
   several consecutive frames, and Kalman smoothing keeps object identities/boxes stable
   when detections briefly flicker or drop out.
2. **Adaptive tracking** - the data-association gate (IoU threshold + track survival age)
   adapts to each object's estimated **speed and direction**. This is especially important
   for **SportsMOT**, where players exhibit fast, variable-speed motion, frequent occlusion
   and similar team uniforms.

### Dataset
**SportsMOT** - https://deeperaction.github.io/datasets/sportsmot.html

A large-scale multi-object-tracking dataset of basketball/football/volleyball clips in which only the
**players** are tracked. Every clip follows the **MOTChallenge** format (`img1/`,
`gt/gt.txt`, `seqinfo.ini`). We are using a small sample of basketball sequence from a hugging face mirror for our assignment.

### Pipeline overview
```
video frames --> preprocessing/augmentation --> Faster R-CNN (fine-tuned) --> detections
             --> temporal-consistency filter --> adaptive SORT (Kalman + Hungarian)
             --> tracked player identities --> evaluation (mAP, MOTA, IDF1, ID-switches, FPS)
```

## 1. Environment Setup
Install the extra tracking & evaluation dependencies (`filterpy` for the Kalman filter, `lap` for fast assignment, `motmetrics` for MOTA/IDF1, `torchmetrics` for mAP).

In [ ]:
import os, sys, subprocess

# Disable the Xet transfer backend (it can stall HF downloads at 0 B on Colab).
# Must be set before huggingface_hub is first imported.
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for mod, pkg in [("filterpy", "filterpy"), ("lap", "lap"),
                 ("motmetrics", "motmetrics"), ("torchmetrics", "torchmetrics"),
                 ("huggingface_hub", "huggingface_hub"), ("cv2", "opencv-python")]:
    try:
        __import__(mod)
    except ImportError:
        print(f"Installing {pkg} ...")
        pip_install([pkg])

# install torch / torchvision if missing.
try:
    import torch, torchvision
except ImportError:
    pip_install(["torch", "torchvision"])
print("Dependencies ready.")

Install other necessary packages

In [ ]:
import os, glob, time, random, configparser
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
import torchvision
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# Choose the best available device.
FORCE_DEVICE = None

def pick_device():
    """Auto-select the compute device based on availability: CUDA -> Intel XPU -> CPU."""
    if FORCE_DEVICE:
        return torch.device(FORCE_DEVICE)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch, "xpu") and torch.xpu.is_available():
        return torch.device("xpu")
    return torch.device("cpu")

DEVICE = pick_device()
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Device:", DEVICE)

## 2. Data Acquisition
We download one small **SportsMOT** clip automatically. Each clip is stored in the
MOTChallenge folder layout (`img1/`, `gt/gt.txt`, `seqinfo.ini`) and every ground-truth box
is a *player*, which keeps preprocessing simple. We fetch a single clip from a Hugging Face
mirror of SportsMOT (downloading only the frames we need) so the notebook stays
self-contained and runs the same on Colab or locally.

> **(Optional) Hugging Face token — faster & fewer warnings.** Anonymous downloads are
> rate-limited. To use a token: create a *read* token at
> https://huggingface.co/settings/tokens, then on Colab open **🔑 Secrets** in the left
> sidebar, add a secret named `HF_TOKEN` with your token as the value, and enable
> *Notebook access*. Locally, set the `HF_TOKEN` environment variable instead. The next cell
> picks it up automatically; without a token it falls back to anonymous access.

In [ ]:
# ----------------------- Download Configuration -----------------------
# Problem Statement 3 dataset: SportsMOT
# (https://deeperaction.github.io/datasets/sportsmot.html).
# SportsMOT stores each clip in MOTChallenge format (img1/, gt/gt.txt, seqinfo.ini)
# and the tracked targets are *players* (a sub-category of COCO's "person" class).
# We pull one clip from a reachable Hugging Face mirror so the notebook is
# self-contained and easy to run on Colab or locally.
HF_REPO   = "Lekim89/sportsmot"    # MOTChallenge-format SportsMOT mirror
SEQUENCE  = "v_-6Os86HzwCs_c001"   # a basketball clip (825 frames, 1280x720)
DATA_ROOT = "data"

MAX_FRAMES   = 450                 # cap frames for fast train/eval (None -> use all)
SCORE_THRESH = 0.70                # detection confidence threshold
EPOCHS       = 10                  # fine-tuning epochs (raise for better accuracy)
BATCH_SIZE   = 2

os.makedirs(DATA_ROOT, exist_ok=True)

Download the frames.

In [ ]:
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

def get_hf_token():
    """Return an HF token from the HF_TOKEN env var or a Colab secret (None if not set)."""
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN = get_hf_token()
print("HF token:", "found" if HF_TOKEN else "not set (using anonymous access)")

def download_sequence_hf(repo, sequence, root, max_frames=None, token=None):
    """Fetch a single SportsMOT clip from a Hugging Face dataset mirror.

    Uses hf_hub_download per file (not snapshot_download): the mirror holds all 240
    clips (~180k images), and snapshot_download would enumerate that whole tree first,
    which times out on Colab. A token (if provided) raises rate limits and avoids the
    anonymous-access warnings; otherwise we pass token=False to skip the secret lookup.
    """
    from huggingface_hub import hf_hub_download
    from concurrent.futures import ThreadPoolExecutor

    def _get(rel_path):
        return hf_hub_download(repo_id=repo, repo_type="dataset",
                               filename=rel_path, token=token or False)

    base = f"train/{sequence}"
    seqinfo_path = _get(f"{base}/seqinfo.ini")
    _get(f"{base}/gt/gt.txt")
    seq_dir = os.path.dirname(seqinfo_path)

    # Decide how many frames to pull from seqinfo's seqLength (clip is 1-indexed).
    cfg = configparser.ConfigParser(); cfg.read(seqinfo_path)
    seq_len = int(cfg["Sequence"].get("seqLength", 0))
    n = min(max_frames, seq_len) if max_frames else seq_len

    print(f"Downloading '{sequence}' from HF dataset '{repo}' (first {n} frames) ...")
    frames = [f"{base}/img1/{i:06d}.jpg" for i in range(1, n + 1)]
    with ThreadPoolExecutor(max_workers=8) as ex:
        list(ex.map(_get, frames))
    print("Sequence ready:", seq_dir)
    return seq_dir

SEQ_DIR = download_sequence_hf(HF_REPO, SEQUENCE, DATA_ROOT, MAX_FRAMES, token=HF_TOKEN)

## 3. Data Preprocessing
We parse `seqinfo.ini` for frame metadata, load the frame paths, and read the
ground-truth annotations (`gt/gt.txt`). MOTChallenge ground truth is stored as
`frame, id, bb_left, bb_top, bb_width, bb_height, conf, [class, visibility]`.
We keep only active player boxes (class 1, `conf==1`).

In [ ]:
def parse_seqinfo(seq_dir):
    """Read seqinfo.ini and return the clip metadata (name, image dir, size, fps, length)."""
    ini = os.path.join(seq_dir, "seqinfo.ini")
    cfg = configparser.ConfigParser()
    cfg.read(ini)
    s = cfg["Sequence"]
    return {
        "name": s.get("name", os.path.basename(seq_dir)),
        "imDir": s.get("imDir", "img1"),
        "frameRate": int(s.get("frameRate", 25)),
        "seqLength": int(s.get("seqLength", 0)),
        "imWidth": int(s.get("imWidth", 0)),
        "imHeight": int(s.get("imHeight", 0)),
        "imExt": s.get("imExt", ".jpg"),
    }

def load_frames(seq_dir, info):
    """Return the sorted list of frame image paths for the clip."""
    img_dir = os.path.join(seq_dir, info["imDir"])
    frames = sorted(glob.glob(os.path.join(img_dir, f"*{info['imExt']}")))
    return frames

def load_gt(seq_dir):
    """Parse gt.txt into {frame_idx(1-based): [(track_id, x1,y1,x2,y2), ...]} for players only."""
    path = os.path.join(seq_dir, "gt", "gt.txt")
    gt = {}
    if not os.path.exists(path):
        print("No gt.txt found (test split?)."); return gt
    data = np.atleast_2d(np.loadtxt(path, delimiter=","))
    for row in data:
        fr, tid = int(row[0]), int(row[1])
        x, y, w, h = row[2], row[3], row[4], row[5]
        conf = row[6] if len(row) > 6 else 1
        cls  = int(row[7]) if len(row) > 7 else 1
        if len(row) > 7 and (conf == 0 or cls != 1):
            continue   # keep only active players (class 1)
        gt.setdefault(fr, []).append((tid, x, y, x + w, y + h))
    return gt

INFO   = parse_seqinfo(SEQ_DIR)
FRAMES = load_frames(SEQ_DIR, INFO)
GT     = load_gt(SEQ_DIR)

# Cap to MAX_FRAMES so training/eval stay fast. keep GT aligned to used frames.
if MAX_FRAMES:
    FRAMES = FRAMES[:MAX_FRAMES]
GT = {f: v for f, v in GT.items() if f <= len(FRAMES)}

print(INFO)
print(f"#frames used = {len(FRAMES)} | #annotated frames = {len(GT)}")

In [ ]:
def show_frame_with_boxes(idx=0):
    """Display one frame with its ground-truth player boxes and track ids drawn on top."""
    img = cv2.cvtColor(cv2.imread(FRAMES[idx]), cv2.COLOR_BGR2RGB)
    for tid, x1, y1, x2, y2 in GT.get(idx + 1, []):
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(img, str(tid), (int(x1), int(y1) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    plt.figure(figsize=(9, 6)); plt.imshow(img)
    plt.title(f"{INFO['name']} - frame {idx + 1} (ground truth)"); plt.axis("off"); plt.show()

show_frame_with_boxes(0)

### 3.1 Detection Dataset & Data Augmentation
We build a `torch.utils.data.Dataset` that yields `(image_tensor, target)` where the
target holds player boxes/labels. Augmentation (**random horizontal flip, random crop,
and color jitter**) improves robustness; boxes are transformed consistently and degenerate
boxes are filtered. Frames are split temporally: first 80% for training, last 20% for
validation/evaluation.

In [ ]:
import torchvision.transforms.functional as TF
from torchvision.transforms import ColorJitter

class Compose:
    """Chain a list of (image, target) transforms into one callable."""
    def __init__(self, tfms): self.tfms = tfms
    def __call__(self, img, tgt):
        for t in self.tfms: img, tgt = t(img, tgt)
        return img, tgt

class RandomHorizontalFlip:
    """Randomly flip the image left-right and mirror the box x-coordinates."""
    def __init__(self, p=0.5): self.p = p
    def __call__(self, img, tgt):
        if random.random() < self.p:
            W = img.width
            img = TF.hflip(img)
            b = tgt["boxes"].clone()
            b[:, [0, 2]] = W - b[:, [2, 0]]
            tgt["boxes"] = b
        return img, tgt

class RandomColorJitter:
    """Randomly perturb brightness/contrast/saturation/hue (boxes unchanged)."""
    def __init__(self, p=0.5):
        self.p = p
        self.jit = ColorJitter(0.2, 0.2, 0.2, 0.05)
    def __call__(self, img, tgt):
        if random.random() < self.p:
            img = self.jit(img)
        return img, tgt

class RandomCrop:
    """Crop a random 80-100% sub-window; keep boxes whose centre stays inside."""
    def __init__(self, p=0.5): self.p = p
    def __call__(self, img, tgt):
        if random.random() > self.p or len(tgt["boxes"]) == 0:
            return img, tgt
        W, H = img.width, img.height
        cw, ch = int(W * random.uniform(0.8, 1.0)), int(H * random.uniform(0.8, 1.0))
        x0, y0 = random.randint(0, W - cw), random.randint(0, H - ch)
        b = tgt["boxes"].clone()
        cx, cy = (b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2
        keep = (cx >= x0) & (cx < x0 + cw) & (cy >= y0) & (cy < y0 + ch)
        if keep.sum() == 0:
            return img, tgt
        b = b[keep]
        b[:, [0, 2]] = b[:, [0, 2]].clamp(x0, x0 + cw) - x0
        b[:, [1, 3]] = b[:, [1, 3]].clamp(y0, y0 + ch) - y0
        img = TF.crop(img, y0, x0, ch, cw)
        tgt = {"boxes": b, "labels": tgt["labels"][keep]}
        return img, tgt

class ToTensor:
    """Convert a PIL image to a float tensor (target passed through)."""
    def __call__(self, img, tgt):
        return TF.to_tensor(img), tgt

def valid_boxes(tgt):
    """Drop degenerate boxes (zero/negative width or height) from a target dict."""
    b = tgt["boxes"]
    keep = (b[:, 2] > b[:, 0]) & (b[:, 3] > b[:, 1])
    return {"boxes": b[keep], "labels": tgt["labels"][keep]}

In [ ]:
class MOTDetectionDataset(torch.utils.data.Dataset):
    """Yields (image_tensor, target) pairs of player boxes for detector training/eval."""
    PLAYER = 1  # single foreground class id
    def __init__(self, frames, gt, indices, transforms=None):
        self.frames, self.gt = frames, gt
        self.indices, self.transforms = indices, transforms
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        """Load frame `indices[i]`, build its target dict, and apply transforms."""
        idx = self.indices[i]
        img = Image.open(self.frames[idx]).convert("RGB")
        boxes = [[x1, y1, x2, y2] for _, x1, y1, x2, y2 in self.gt.get(idx + 1, [])]
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        target = {"boxes": boxes,
                  "labels": torch.ones((len(boxes),), dtype=torch.int64)}
        if self.transforms:
            img, target = self.transforms(img, target)
        else:
            img = TF.to_tensor(img)
        return img, valid_boxes(target)

def collate_fn(batch):
    """Collate variable-length detection samples by keeping them as tuples (no stacking)."""
    return tuple(zip(*batch))

n = len(FRAMES)
split = int(n * 0.8)
train_idx = list(range(0, split))
val_idx   = list(range(split, n))

train_tfms = Compose([RandomHorizontalFlip(0.5), RandomCrop(0.4),
                      RandomColorJitter(0.5), ToTensor()])
train_ds = MOTDetectionDataset(FRAMES, GT, train_idx, train_tfms)
val_ds   = MOTDetectionDataset(FRAMES, GT, val_idx, Compose([ToTensor()]))

# num_workers=0 for cross-platform safety: on Windows notebook kernels, worker
# subprocesses cannot pickle the locally-defined dataset/transform classes.
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                           collate_fn=collate_fn, num_workers=0)
print(f"train frames = {len(train_ds)} | val frames = {len(val_ds)}")

## 4. Model Development — Faster R-CNN
We start from a **COCO-pretrained `fasterrcnn_resnet50_fpn`** and replace the box
predictor head with a 2-class head (`background`, `player`). Transfer learning from
COCO (which already knows the *person* class) lets us fine-tune effectively on a small
sequence. We keep the **COCO-pretrained model as a baseline detector** for comparison.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

COCO_PERSON = 1  # 'person' label id in COCO

def build_finetune_model(num_classes=2):
    """Load a COCO-pretrained Faster R-CNN and swap in a fresh `num_classes` box head."""
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
    return model

# Baseline: the untouched COCO detector (person detections only)
baseline_model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT).to(DEVICE).eval()

# Fine-tuned model (background + player)
model = build_finetune_model(num_classes=2).to(DEVICE)
print("Models built.")

### 4.1 Fine-tuning
We fine-tune with SGD (momentum) and a step LR schedule for a few epochs. Loss is the sum
of the Faster R-CNN RPN and ROI losses returned by torchvision in training mode.

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=5e-4)
lr_sched  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

history = []
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for images, targets in train_loader:
        images = [im.to(DEVICE) for im in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
        # skip batches with no boxes
        if all(len(t["boxes"]) == 0 for t in targets):
            continue
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        running += loss.item()
    lr_sched.step()
    avg = running / max(1, len(train_loader))
    history.append(avg)
    print(f"Epoch {epoch + 1}/{EPOCHS} - mean loss {avg:.4f}")

plt.plot(range(1, len(history) + 1), history, marker="o")
plt.title("Fine-tuning loss"); plt.xlabel("epoch"); plt.ylabel("mean loss")
plt.grid(True); plt.show()
model.eval(); print("Fine-tuning complete.")

## 5. Detection Evaluation (mAP)
We run both detectors on the held-out validation frames and compute **mean Average
Precision** with a small, self-contained implementation (COCO-style 101-point
interpolation, no `pycocotools` dependency). Boxes are filtered to the *person* class.
This isolates detector quality before tracking.

In [ ]:
@torch.no_grad()
def detect(model_, img_tensor, person_label, score_thresh=SCORE_THRESH):
    """Run the detector on one image and return (boxes_xyxy[np], scores[np]) for the target class."""
    out = model_([img_tensor.to(DEVICE)])[0]
    keep = (out["labels"] == person_label) & (out["scores"] >= score_thresh)
    return (out["boxes"][keep].cpu().numpy(),
            out["scores"][keep].cpu().numpy())

# --- self-contained mAP (no pycocotools/torchmetrics needed) --------------
def _iou_one_to_many(box, boxes):
    """IoU of a single box[4] against an array of boxes[M,4], returned as [M]."""
    if len(boxes) == 0:
        return np.zeros((0,), dtype=np.float32)
    xx1 = np.maximum(box[0], boxes[:, 0]); yy1 = np.maximum(box[1], boxes[:, 1])
    xx2 = np.minimum(box[2], boxes[:, 2]); yy2 = np.minimum(box[3], boxes[:, 3])
    w = np.clip(xx2 - xx1, 0, None); h = np.clip(yy2 - yy1, 0, None)
    inter = w * h
    area = (box[2] - box[0]) * (box[3] - box[1])
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return inter / (area + areas - inter + 1e-9)

def _ap_at_iou(preds, gts, iou_thr):
    """Average Precision at one IoU threshold. preds: per-image (boxes[N,4], scores[N]); gts: boxes[M,4]."""
    dets = [(s[j], i, b[j]) for i, (b, s) in enumerate(preds) for j in range(len(b))]
    n_gt = sum(len(g) for g in gts)
    if n_gt == 0:
        return 0.0
    dets.sort(key=lambda x: -x[0])
    matched = [np.zeros(len(g), dtype=bool) for g in gts]
    tp = np.zeros(len(dets)); fp = np.zeros(len(dets))
    for k, (sc, i, box) in enumerate(dets):
        ious = _iou_one_to_many(box, gts[i])
        if len(ious) and ious.max() >= iou_thr and not matched[i][int(ious.argmax())]:
            tp[k] = 1; matched[i][int(ious.argmax())] = True
        else:
            fp[k] = 1
    tp_c, fp_c = np.cumsum(tp), np.cumsum(fp)
    recall = tp_c / (n_gt + 1e-9)
    precision = tp_c / (tp_c + fp_c + 1e-9)
    ap = 0.0                                   # COCO 101-point interpolation
    for t in np.linspace(0, 1, 101):
        p = precision[recall >= t].max() if np.any(recall >= t) else 0.0
        ap += p / 101
    return float(ap)

def eval_detection_map(model_, person_label, name):
    """Compute mAP@0.5 and mAP@[.5:.95] for a detector over the validation frames."""
    model_.eval()
    preds, gts = [], []
    for idx in val_idx:
        img = TF.to_tensor(Image.open(FRAMES[idx]).convert("RGB"))
        boxes, scores = detect(model_, img, person_label, score_thresh=0.05)
        preds.append((boxes, scores))
        gts.append(np.array([[x1, y1, x2, y2]
                             for _, x1, y1, x2, y2 in GT.get(idx + 1, [])]).reshape(-1, 4))
    ap50 = _ap_at_iou(preds, gts, 0.5)
    mAP  = float(np.mean([_ap_at_iou(preds, gts, t) for t in np.arange(0.5, 1.0, 0.05)]))
    print(f"[{name}] mAP@0.50:0.95 = {mAP:.3f} | mAP@0.50 = {ap50:.3f}")
    return {"map": mAP, "map_50": ap50}

map_baseline  = eval_detection_map(baseline_model, COCO_PERSON, "Baseline (COCO)")
map_finetuned = eval_detection_map(model, MOTDetectionDataset.PLAYER, "Fine-tuned")

## 6. Tracking — Adaptive SORT with Temporal Consistency
We implement **SORT** (Simple Online and Realtime Tracking): each track is a **Kalman
filter** in `[cx, cy, area, aspect]` state space with constant-velocity motion; detections
are matched to predicted tracks by **IoU + Hungarian assignment**.

**Novel enhancements added on top of vanilla SORT:**

* **Temporal consistency check** — a track must accumulate `min_hits` detections in
  consecutive frames before its identity is emitted, which removes one-frame false
  positives; the Kalman filter also *coasts* a track for up to `max_age` frames when a
  detection is momentarily missed, keeping IDs stable.
* **Adaptive tracking** — the association gate is a function of each track's estimated
  **speed** (Kalman velocity magnitude). Fast objects relax their IoU threshold and get a
  longer survival age, so large frame-to-frame displacements still match; slow objects stay
  tightly gated to avoid ID swaps.

In [ ]:
from filterpy.kalman import KalmanFilter
from scipy.optimize import linear_sum_assignment

def iou_batch(a, b):
    """IoU between every box in a[N,4] and b[M,4] (xyxy), returned as an [N,M] matrix."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a, b = a[:, None, :], b[None, :, :]
    xx1 = np.maximum(a[..., 0], b[..., 0]); yy1 = np.maximum(a[..., 1], b[..., 1])
    xx2 = np.minimum(a[..., 2], b[..., 2]); yy2 = np.minimum(a[..., 3], b[..., 3])
    w = np.clip(xx2 - xx1, 0, None); h = np.clip(yy2 - yy1, 0, None)
    inter = w * h
    area_a = (a[..., 2] - a[..., 0]) * (a[..., 3] - a[..., 1])
    area_b = (b[..., 2] - b[..., 0]) * (b[..., 3] - b[..., 1])
    return inter / (area_a + area_b - inter + 1e-9)

def xyxy_to_z(box):
    """Convert an xyxy box to Kalman measurement [cx, cy, area, aspect-ratio] (4x1)."""
    w = box[2] - box[0]; h = box[3] - box[1]
    cx = box[0] + w / 2.; cy = box[1] + h / 2.
    s = w * h; r = w / (h + 1e-6)
    return np.array([cx, cy, s, r]).reshape(4, 1)

def z_to_xyxy(x):
    """Convert a Kalman state [cx, cy, area, aspect, ...] back to an xyxy box."""
    cx, cy, s, r = x[0], x[1], x[2], x[3]
    w = np.sqrt(abs(s) * abs(r)); h = s / (w + 1e-6)
    return np.array([cx - w / 2., cy - h / 2., cx + w / 2., cy + h / 2.]).reshape(-1)

In [ ]:
class KalmanBoxTracker:
    """A single track: a constant-velocity Kalman filter over one player's box."""
    count = 0
    def __init__(self, box):
        """Initialise the 7-state filter (cx, cy, area, aspect, +velocities) from an xyxy box."""
        self.kf = KalmanFilter(dim_x=7, dim_z=4)
        self.kf.F = np.array([[1,0,0,0,1,0,0],[0,1,0,0,0,1,0],[0,0,1,0,0,0,1],
                              [0,0,0,1,0,0,0],[0,0,0,0,1,0,0],[0,0,0,0,0,1,0],
                              [0,0,0,0,0,0,1]], dtype=float)
        self.kf.H = np.array([[1,0,0,0,0,0,0],[0,1,0,0,0,0,0],
                              [0,0,1,0,0,0,0],[0,0,0,1,0,0,0]], dtype=float)
        self.kf.R[2:, 2:] *= 10.
        self.kf.P[4:, 4:] *= 1000.; self.kf.P *= 10.
        self.kf.Q[-1, -1] *= 0.01; self.kf.Q[4:, 4:] *= 0.01
        self.kf.x[:4] = xyxy_to_z(box)
        self.id = KalmanBoxTracker.count; KalmanBoxTracker.count += 1
        self.time_since_update = 0
        self.hits = 0; self.hit_streak = 0; self.age = 0
    def update(self, box):
        """Correct the filter with a matched detection and refresh the hit counters."""
        self.time_since_update = 0
        self.hits += 1; self.hit_streak += 1
        self.kf.update(xyxy_to_z(box))
    def predict(self):
        """Advance the filter one frame and return the predicted xyxy box."""
        if self.kf.x[6] + self.kf.x[2] <= 0:
            self.kf.x[6] *= 0.0
        self.kf.predict()
        self.age += 1
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        return z_to_xyxy(self.kf.x)
    def get_state(self):
        """Return the current estimated box as xyxy."""
        return z_to_xyxy(self.kf.x)
    def speed(self):
        """Return the estimated speed |velocity| in pixels/frame (kf.x is a (7,1) column vector)."""
        return float(np.hypot(self.kf.x[4, 0], self.kf.x[5, 0]))

In [ ]:
class AdaptiveSort:
    """SORT tracker with speed-adaptive gating + temporal-consistency confirmation."""
    def __init__(self, max_age=15, min_hits=3, iou_base=0.3, adaptive=True):
        self.max_age = max_age; self.min_hits = min_hits
        self.iou_base = iou_base; self.adaptive = adaptive
        self.trackers = []; self.frame_count = 0

    def _gate(self, trk):
        """Return the IoU match threshold for a track (lower for faster tracks when adaptive)."""
        if not self.adaptive:
            return self.iou_base
        return max(0.1, self.iou_base - 0.02 * min(trk.speed(), 10.0))

    def update(self, dets):
        """Advance all tracks one frame with the new detections; return confirmed [x1,y1,x2,y2,id] rows."""
        self.frame_count += 1
        preds = np.array([t.predict() for t in self.trackers]).reshape(-1, 4)
        matches, un_dets, un_trks = self._associate(dets, preds)
        for d, t in matches:
            self.trackers[t].update(dets[d])
        for d in un_dets:
            self.trackers.append(KalmanBoxTracker(dets[d]))
        out = []
        for t in reversed(range(len(self.trackers))):
            trk = self.trackers[t]
            # adaptive survival: fast tracks may coast a little longer
            eff_age = self.max_age + (5 if self.adaptive and trk.speed() > 5 else 0)
            if trk.time_since_update > eff_age:
                self.trackers.pop(t); continue
            # temporal-consistency: emit only confirmed tracks
            if trk.time_since_update < 1 and (
                    trk.hit_streak >= self.min_hits or self.frame_count <= self.min_hits):
                box = trk.get_state()
                out.append(np.concatenate([box, [trk.id + 1]]))
        return np.array(out).reshape(-1, 5)

    def _associate(self, dets, preds):
        """Match detections to predicted tracks via IoU + Hungarian; return (matches, unmatched_dets, unmatched_trks)."""
        if len(preds) == 0:
            return [], list(range(len(dets))), []
        if len(dets) == 0:
            return [], [], list(range(len(preds)))
        iou = iou_batch(dets, preds)
        row, col = linear_sum_assignment(-iou)
        matches, un_dets, un_trks = [], [], []
        matched_d, matched_t = set(), set()
        for d, t in zip(row, col):
            if iou[d, t] >= self._gate(self.trackers[t]):
                matches.append((d, t)); matched_d.add(d); matched_t.add(t)
        un_dets = [d for d in range(len(dets)) if d not in matched_d]
        un_trks = [t for t in range(len(preds)) if t not in matched_t]
        return matches, un_dets, un_trks

### 6.1 Baseline tracker (for comparison)
A simple **greedy IoU tracker** without a Kalman filter, adaptive gating, or temporal
confirmation. Comparing against it isolates the benefit of our enhancements.

In [ ]:
class GreedyIouTracker:
    """Baseline tracker: greedy IoU matching, no Kalman filter / adaptive gating / confirmation."""
    def __init__(self, iou_thresh=0.3, max_age=1):
        self.iou_thresh = iou_thresh; self.max_age = max_age
        self.tracks = []  # each: {box, id, age}
        self.next_id = 1
    def update(self, dets):
        """Greedily assign detections to existing tracks by IoU; return [x1,y1,x2,y2,id] rows."""
        preds = np.array([t["box"] for t in self.tracks]).reshape(-1, 4)
        iou = iou_batch(dets, preds)
        assigned = set(); out = []
        for d in range(len(dets)):
            j = int(np.argmax(iou[d])) if len(preds) else -1
            if j >= 0 and j not in assigned and iou[d, j] >= self.iou_thresh:
                self.tracks[j].update(box=dets[d], age=0); assigned.add(j)
                out.append(np.concatenate([dets[d], [self.tracks[j]["id"]]]))
            else:
                self.tracks.append({"box": dets[d], "id": self.next_id, "age": 0})
                out.append(np.concatenate([dets[d], [self.next_id]])); self.next_id += 1
        for j, t in enumerate(self.tracks):
            if j not in assigned:
                t["age"] += 1
        self.tracks = [t for t in self.tracks if t["age"] <= self.max_age]
        return np.array(out).reshape(-1, 5)

## 7. Run the Tracking Pipeline
We run the full sequence through the fine-tuned detector and each tracker, storing results
in **MOT format** (`frame, id, x, y, w, h`) for evaluation, and measuring detection+tracking
**speed (FPS)**.

In [ ]:
def run_tracking(detector, person_label, tracker, frames, save_video=None):
    """Detect + track over all frames; return ({frame: [(id,x1,y1,x2,y2)]}, fps) and optionally write a video."""
    KalmanBoxTracker.count = 0
    results = {}   # frame_idx(1-based) -> list[(id, x1,y1,x2,y2)]
    writer = None
    t0 = time.time()
    for idx, fpath in enumerate(frames):
        img_bgr = cv2.imread(fpath)
        img_t = TF.to_tensor(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        boxes, scores = detect(detector, img_t, person_label)
        tracks = tracker.update(boxes.astype(float))
        frame_res = []
        for tr in tracks:
            x1, y1, x2, y2, tid = tr
            frame_res.append((int(tid), x1, y1, x2, y2))
            if save_video is not None:
                c = (int(37 * tid) % 255, int(17 * tid) % 255, int(29 * tid) % 255)
                cv2.rectangle(img_bgr, (int(x1), int(y1)), (int(x2), int(y2)), c, 2)
                cv2.putText(img_bgr, f"ID {int(tid)}", (int(x1), int(y1) - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 2)
        results[idx + 1] = frame_res
        if save_video is not None:
            if writer is None:
                h, w = img_bgr.shape[:2]
                writer = cv2.VideoWriter(save_video, cv2.VideoWriter_fourcc(*"mp4v"),
                                         INFO["frameRate"], (w, h))
            writer.write(img_bgr)
    if writer is not None:
        writer.release()
    fps = len(frames) / (time.time() - t0)
    return results, fps

# Proposed system: fine-tuned detector + adaptive SORT (temporal consistency ON)
res_proposed, fps_proposed = run_tracking(
    model, MOTDetectionDataset.PLAYER,
    AdaptiveSort(max_age=15, min_hits=3, adaptive=True),
    FRAMES, save_video="tracking_output.mp4")

# Baseline system: COCO detector + greedy IoU tracker
res_baseline, fps_baseline = run_tracking(
    baseline_model, COCO_PERSON, GreedyIouTracker(), FRAMES)

print(f"Proposed FPS = {fps_proposed:.2f} | Baseline FPS = {fps_baseline:.2f}")

In [ ]:
def preview_tracked(results, sample_frames=(0, len(FRAMES)//2, len(FRAMES)-1)):
    """Draw the tracked player boxes/ids for a few sample frames side by side."""
    fig, axes = plt.subplots(1, len(sample_frames), figsize=(16, 5))
    for ax, idx in zip(axes, sample_frames):
        img = cv2.cvtColor(cv2.imread(FRAMES[idx]), cv2.COLOR_BGR2RGB)
        for tid, x1, y1, x2, y2 in results.get(idx + 1, []):
            c = ((37 * tid) % 255, (17 * tid) % 255, (29 * tid) % 255)
            cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), c, 2)
            cv2.putText(img, f"{tid}", (int(x1), int(y1) - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 2)
        ax.imshow(img); ax.set_title(f"frame {idx + 1}"); ax.axis("off")
    plt.suptitle("Proposed system - tracked identities"); plt.tight_layout(); plt.show()

preview_tracked(res_proposed)

## 8. Tracking Evaluation (MOTA, IDF1, ID switches)
We evaluate against the ground truth with `motmetrics`. Distances use IoU (max 0.5).
Key metrics:
* **MOTA** — overall accuracy (1 − (FN+FP+IDsw)/GT); higher is better.
* **IDF1** — identity-preservation F1; rewards keeping the *same* ID on an object.
* **ID switches** — how often a tracked identity jumps between objects (lower is better).
* **Precision / Recall** — detection quality within the tracker output.

In [ ]:
import motmetrics as mm
import pandas as pd

# motmetrics 1.4 still calls np.asfarray, which was removed in NumPy 2.0
# (Colab ships NumPy 2.x). Restore a minimal shim so evaluation works.
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=np.float64: np.asarray(a, dtype=dtype)

def evaluate_tracking(results, gt, name):
    """Score tracker output against ground truth (IoU<=0.5 matching) and return a MOT metrics row."""
    acc = mm.MOTAccumulator(auto_id=True)
    for fr in sorted(gt.keys()):
        gt_ids  = [t for (t, *_ ) in gt[fr]]
        gt_xywh = [[x1, y1, x2 - x1, y2 - y1] for (_, x1, y1, x2, y2) in gt[fr]]
        hyp = results.get(fr, [])
        hyp_ids  = [t for (t, *_ ) in hyp]
        hyp_xywh = [[x1, y1, x2 - x1, y2 - y1] for (_, x1, y1, x2, y2) in hyp]
        dists = mm.distances.iou_matrix(gt_xywh, hyp_xywh, max_iou=0.5)
        acc.update(gt_ids, hyp_ids, dists)
    mh = mm.metrics.create()
    summary = mh.compute(
        acc, name=name,
        metrics=["num_frames", "mota", "motp", "idf1",
                 "num_switches", "num_false_positives", "num_misses",
                 "precision", "recall"])
    return summary

sum_proposed = evaluate_tracking(res_proposed, GT, "Proposed (FRCNN+AdaptiveSORT)")
sum_baseline = evaluate_tracking(res_baseline, GT, "Baseline (COCO+GreedyIoU)")
track_summary = pd.concat([sum_baseline, sum_proposed])
print(mm.io.render_summary(
    track_summary,
    formatters=mm.metrics.create().formatters,
    namemap=mm.io.motchallenge_metric_names))
track_summary

## 9. Consolidated Results

In [ ]:
results_table = pd.DataFrame({
    "System":     ["Baseline (COCO + Greedy IoU)", "Proposed (Fine-tuned + Adaptive SORT)"],
    "mAP@0.5":    [round(map_baseline["map_50"], 3), round(map_finetuned["map_50"], 3)],
    "mAP@.5:.95": [round(map_baseline["map"], 3),    round(map_finetuned["map"], 3)],
    "MOTA":       [round(sum_baseline["mota"].iloc[0], 3),
                   round(sum_proposed["mota"].iloc[0], 3)],
    "IDF1":       [round(sum_baseline["idf1"].iloc[0], 3),
                   round(sum_proposed["idf1"].iloc[0], 3)],
    "ID switches":[int(sum_baseline["num_switches"].iloc[0]),
                   int(sum_proposed["num_switches"].iloc[0])],
    "FPS":        [round(fps_baseline, 2), round(fps_proposed, 2)],
})
display(results_table)

ax = results_table.set_index("System")[["mAP@0.5", "MOTA", "IDF1"]].plot(
    kind="bar", figsize=(9, 4), rot=0)
ax.set_title("Baseline vs Proposed"); ax.set_ylabel("score"); plt.tight_layout(); plt.show()

## 10. Justification & Analysis

**Why Faster R-CNN?** It is a two-stage detector with a Region Proposal Network that gives
high localisation accuracy for small/occluded targets — well suited to crowded sports
scenes where players overlap heavily. We fine-tuned from COCO weights because SportsMOT
players are a sub-category of COCO's *person* class, so transfer learning converges in a
few epochs on a small clip.

**Observed result — why the baseline collapses.** In our run the baseline (COCO detector +
greedy IoU) records a **negative MOTA** with thousands of false positives, while the
proposed system reaches **MOTA ≈ 0.96 / IDF1 ≈ 0.90**. The reason is domain-specific: the
raw COCO detector fires on *every* person — players, referees, coaches and the crowd in the
stands — but SportsMOT ground truth annotates **only the on-court players**. Every
spectator box therefore counts as a false positive. Fine-tuning teaches the detector the
task-specific definition of the target (players, not spectators), which is the single
biggest driver of the improvement (precision jumps from ~0.30 to ~1.00).

**Effect of fine-tuning (detection mAP).** Consistent with the above, fine-tuning lifts
mAP@0.5 well above the COCO baseline because it adapts to this clip's camera motion, player
scale and court background. Risks to watch: **overfitting** to the first 80% of frames — the
temporal split means train/val come from different parts of the same rally, causing domain
shift — or **underfitting** from too few epochs. Increasing `EPOCHS` and using more clips
would address both.

**Effect of the tracker (MOTA / IDF1 / ID-switches).** On top of the better detector, the
proposed **adaptive SORT** beats a greedy IoU tracker on **IDF1** and **ID switches**
because:
* the **Kalman filter** predicts motion, so associations survive short detection gaps caused
  by player-on-player occlusion;
* the **temporal-consistency** `min_hits` gate suppresses one-frame false positives that
  otherwise spawn spurious IDs;
* **adaptive gating** widens the match region for fast-moving players — SportsMOT is
  characterised by rapid, variable-speed motion, so this directly prevents the broken tracks
  that show up as ID switches. Similar team uniforms make appearance cues weak, which is
  exactly why motion-based gating matters here.

**Speed (FPS).** The two-stage detector dominates runtime, so both systems run at similar
FPS; the tracker adds negligible overhead. This is the classic **accuracy-vs-speed**
trade-off — a one-stage detector (YOLO/RetinaNet) would raise FPS at some cost to mAP.

**Limitations.** Single short clip, IoU-only association (no appearance/Re-ID features, so
long occlusions and uniform similarity still cause switches), and evaluation on a temporal
hold-out rather than unseen clips. Deep-SORT-style appearance embeddings and multi-clip
training are the natural next steps.

## 11. Conclusion & Deliverables

We built an end-to-end multi-object tracking system that:
1. **preprocesses & augments** SportsMOT video frames (resize/normalize via tensor conversion,
   random flip/crop/color-jitter);
2. **detects** players with a COCO-pretrained **Faster R-CNN fine-tuned** on the sequence;
3. **tracks** them with an **adaptive SORT** tracker featuring **temporal-consistency checks**
   and **speed-adaptive association**;
4. is **evaluated** with detection mAP and tracking MOTA / IDF1 / ID-switches / FPS against a
   baseline, with analysis of the results.

**Video explanation (≤ 4 min):** _<add your video link here and in the shared Google Sheet>_